# 🏭 Część 5: Production Stack - Kompletny przewodnik

**Cel:** Zrozumieć pełny production stack od A do Z, SSL, monitoring, best practices.

**To jest podsumowanie wszystkich poprzednich notebooków + dodatki production.**

---

## 1. Pełny Production Stack - architektura

### Typowy stack dla FastAPI w produkcji:

```
┌──────────────────────────────────────────────────────────────┐
│                         INTERNET                              │
└────────────────────┬─────────────────────────────────────────┘
                     │
                     ↓ HTTPS (port 443)
              ┌──────────────┐
              │    nginx     │  Warstwa 1: HTTP Server / Reverse Proxy
              │ (port 80/443)│  - SSL termination (Let's Encrypt)
              │              │  - Static files (/static/, /media/)
              │              │  - Load balancing
              │              │  - Gzip compression
              │              │  - Rate limiting
              └──────┬───────┘
                     │
                     ↓ HTTP (localhost:8000)
         ┌───────────┴───────────┐
         │                       │
    ┌────▼─────┐         ┌──────▼────┐
    │ Uvicorn 1│         │ Uvicorn 2 │  Warstwa 2: ASGI Server
    │ (worker 1)│         │ (worker 2)│  - ASGI interface
    │ (process) │         │ (process) │  - Async request handling
    └────┬─────┘         └──────┬────┘  - Python execution
         │                      │
         └──────────┬───────────┘
                    ↓
            ┌───────────────┐
            │   FastAPI     │         Warstwa 3: Application Framework
            │  (Twój kod)   │         - Routing
            │               │         - Business logic
            │               │         - Pydantic validation
            └───────┬───────┘         - Authentication/Authorization
                    │
                    ↓
            ┌───────────────┐
            │  PostgreSQL   │         Warstwa 4: Database
            │   (Docker)    │         - Persystencja danych
            │               │         - Postgres 15
            └───────┬───────┘
                    │
         ┌──────────┴──────────┐
         │                     │
    ┌────▼────┐         ┌─────▼────┐
    │  Redis  │         │  Sentry  │  Warstwa 5: Dodatkowe serwisy
    │ (cache) │         │(errors)  │  - Redis: caching, sessions
    └─────────┘         └──────────┘  - Sentry: error tracking
```

---

## 2. Flow pojedynczego requesta - krok po kroku

### Request: `GET https://example.com/api/users/123`

```
1. KLIENT (Browser/Mobile)
   ↓ HTTPS request (encrypted)
   GET https://example.com/api/users/123
   
2. nginx (port 443) - WARSTWA 1
   ├─ Odbiera HTTPS
   ├─ Dekryptuje (SSL termination)
   ├─ Sprawdza: to API? (nie /static/)
   ├─ Rate limiting: OK (nie przekroczony limit)
   └─ Przekazuje do Uvicorn:
      ↓ HTTP request (plain, localhost)
      GET http://127.0.0.1:8000/api/users/123
   
3. Uvicorn Worker 1 (port 8000) - WARSTWA 2
   ├─ Odbiera HTTP request
   ├─ Tłumaczy na ASGI:
   │  scope = {
   │      'type': 'http',
   │      'method': 'GET',
   │      'path': '/api/users/123',
   │      ...
   │  }
   └─ Wywołuje: await app(scope, receive, send)
      ↓
   
4. FastAPI (Twój kod) - WARSTWA 3
   ├─ Routing: GET /api/users/{user_id}
   ├─ Dependency injection: get_db()
   ├─ Authentication check (JWT)
   ├─ Authorization check (permissions)
   └─ Wywołuje endpoint:
      ↓
      @app.get("/api/users/{user_id}")
      async def get_user(user_id: int, db = Depends(get_db)):
          user = await db.query(User).filter(id=user_id).first()
          return user
      ↓
   
5. PostgreSQL - WARSTWA 4
   ├─ Wykonuje query: SELECT * FROM users WHERE id = 123
   ├─ Zwraca dane: {id: 123, name: "John", email: "john@example.com"}
   └─ ↓
   
6. FastAPI (powrót)
   ├─ Pydantic serialization (User → JSON)
   ├─ Response: {"id": 123, "name": "John", "email": "john@example.com"}
   └─ Zwraca do Uvicorn
      ↓
   
7. Uvicorn (powrót)
   ├─ Tłumaczy z ASGI na HTTP response:
   │  HTTP/1.1 200 OK
   │  Content-Type: application/json
   │  {"id": 123, "name": "John", "email": "john@example.com"}
   └─ Wysyła do nginx
      ↓
   
8. nginx (powrót)
   ├─ Kompresuje response (gzip)
   ├─ Enkryptuje (HTTPS)
   ├─ Dodaje headers (CORS, security)
   └─ Wysyła do klienta
      ↓ HTTPS response (encrypted)
   
9. KLIENT (Browser/Mobile)
   └─ Otrzymuje JSON: {"id": 123, "name": "John", "email": "john@example.com"}
```

**Czas całkowity:** ~50-150ms (zależnie od DB query)

---

## 3. docker-compose.yml - pełny stack

### Kompletny docker-compose production:


In [ ]:
# docker-compose.yml

version: '3.8'

services:
  # ========================================================================
  # PostgreSQL Database
  # ========================================================================
  db:
    image: postgres:15-alpine
    environment:
      POSTGRES_USER: ${POSTGRES_USER:-fastapi_user}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
      POSTGRES_DB: ${POSTGRES_DB:-fastapi_db}
    volumes:
      - postgres_data:/var/lib/postgresql/data
    restart: unless-stopped
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U fastapi_user"]
      interval: 10s
      timeout: 5s
      retries: 5
    # NIE expose portu 5432 na host (bezpieczeństwo)
    # Dostępny tylko wewnątrz Docker network

  # ========================================================================
  # Redis (Cache, Sessions)
  # ========================================================================
  redis:
    image: redis:7-alpine
    restart: unless-stopped
    volumes:
      - redis_data:/data
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 10s
      timeout: 3s
      retries: 5

  # ========================================================================
  # FastAPI (Uvicorn)
  # ========================================================================
  web:
    build: .
    command: uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
    # LUB z Gunicorn + Uvicorn workers:
    # command: gunicorn main:app --workers 4 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000
    
    depends_on:
      db:
        condition: service_healthy
      redis:
        condition: service_healthy
    
    environment:
      DATABASE_URL: postgresql://${POSTGRES_USER}:${POSTGRES_PASSWORD}@db:5432/${POSTGRES_DB}
      REDIS_URL: redis://redis:6379/0
      SECRET_KEY: ${SECRET_KEY}
      ENVIRONMENT: production
    
    restart: unless-stopped
    
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    
    # NIE expose 8000 na host (nginx będzie proxy)

  # ========================================================================
  # nginx (Reverse Proxy)
  # ========================================================================
  nginx:
    image: nginx:alpine
    ports:
      - "80:80"      # HTTP
      - "443:443"    # HTTPS
    volumes:
      - ./nginx/nginx.conf:/etc/nginx/nginx.conf:ro
      - ./nginx/conf.d:/etc/nginx/conf.d:ro
      - ./static:/var/www/static:ro  # Static files
      - ./media:/var/www/media:ro    # Media files
      - ./certbot/conf:/etc/letsencrypt:ro  # SSL certificates
      - ./certbot/www:/var/www/certbot:ro   # Certbot webroot
    depends_on:
      - web
    restart: unless-stopped

  # ========================================================================
  # Certbot (SSL Certificates)
  # ========================================================================
  certbot:
    image: certbot/certbot
    volumes:
      - ./certbot/conf:/etc/letsencrypt
      - ./certbot/www:/var/www/certbot
    entrypoint: "/bin/sh -c 'trap exit TERM; while :; do certbot renew; sleep 12h & wait $${!}; done;'"

volumes:
  postgres_data:
  redis_data:

### Kluczowe elementy:

1. **healthchecks** - automatyczne sprawdzanie czy serwisy działają
2. **restart: unless-stopped** - auto-restart przy crashu
3. **depends_on + condition** - czeka aż DB jest healthy
4. **NIE expose DB/Redis** - dostępne tylko wewnątrz Docker network (bezpieczeństwo)
5. **Environment variables** - wszystko konfigurowane przez .env
6. **volumes** - persystencja danych

---

## 4. SSL/HTTPS z Let's Encrypt

### Krok 1: Pierwsze uruchomienie (HTTP tylko)

```nginx
# nginx/conf.d/app.conf (initial)

server {
    listen 80;
    server_name example.com www.example.com;

    # Certbot webroot dla weryfikacji
    location /.well-known/acme-challenge/ {
        root /var/www/certbot;
    }

    # Tymczasowo wszystko proxy do app
    location / {
        proxy_pass http://web:8000;
    }
}
```

---

### Krok 2: Uzyskaj certyfikat SSL

```bash
# Uruchom stack
docker-compose up -d

# Uzyskaj certyfikat (wymienia nginx na chwilę)
docker-compose run --rm certbot certonly \
  --webroot \
  --webroot-path=/var/www/certbot \
  -d example.com \
  -d www.example.com \
  --email your@email.com \
  --agree-tos \
  --no-eff-email
```

---

### Krok 3: Konfiguracja HTTPS

```nginx
# nginx/conf.d/app.conf (final)

# Redirect HTTP → HTTPS
server {
    listen 80;
    server_name example.com www.example.com;

    location /.well-known/acme-challenge/ {
        root /var/www/certbot;
    }

    location / {
        return 301 https://$server_name$request_uri;
    }
}

# HTTPS
server {
    listen 443 ssl http2;
    server_name example.com www.example.com;

    # SSL Certificates
    ssl_certificate /etc/letsencrypt/live/example.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/example.com/privkey.pem;

    # SSL Configuration (strong security)
    ssl_protocols TLSv1.2 TLSv1.3;
    ssl_prefer_server_ciphers on;
    ssl_ciphers 'ECDHE-ECDSA-AES128-GCM-SHA256:ECDHE-RSA-AES128-GCM-SHA256';

    # Static files
    location /static/ {
        alias /var/www/static/;
        expires 30d;
        add_header Cache-Control "public, immutable";
    }

    # API proxy
    location / {
        proxy_pass http://web:8000;
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto https;
        
        # Timeouts
        proxy_connect_timeout 60s;
        proxy_send_timeout 60s;
        proxy_read_timeout 60s;
    }
}
```

---

### Auto-renewal certyfikatów

Certbot w docker-compose automatycznie odnawia co 12h (certyfikaty ważne 90 dni).

```bash
# Manualnie force renewal (test)
docker-compose run --rm certbot renew --dry-run
```

---

## 5. Health Checks - endpoint sprawdzający

### FastAPI health check endpoint:


In [ ]:
# main.py

from fastapi import FastAPI, status
from sqlalchemy import text

app = FastAPI()

@app.get("/health", status_code=status.HTTP_200_OK)
async def health_check(db: Session = Depends(get_db)):
    """
    Health check endpoint - sprawdza czy aplikacja działa
    
    Używany przez:
    - Docker healthcheck
    - Load balancers (nginx, AWS ALB)
    - Monitoring (Prometheus, Datadog)
    """
    health_status = {
        "status": "healthy",
        "version": "1.0.0",
        "checks": {}
    }
    
    # Sprawdź połączenie z bazą danych
    try:
        db.execute(text("SELECT 1"))
        health_status["checks"]["database"] = "healthy"
    except Exception as e:
        health_status["status"] = "unhealthy"
        health_status["checks"]["database"] = f"unhealthy: {str(e)}"
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="Database connection failed"
        )
    
    # Sprawdź Redis (opcjonalnie)
    try:
        redis_client.ping()
        health_status["checks"]["redis"] = "healthy"
    except Exception as e:
        health_status["checks"]["redis"] = f"unhealthy: {str(e)}"
        # Redis nie krytyczny - nie zwracaj 503
    
    return health_status

### Response:

```json
{
  "status": "healthy",
  "version": "1.0.0",
  "checks": {
    "database": "healthy",
    "redis": "healthy"
  }
}
```

---

## 6. Monitoring i Logging

### 1. Structured Logging

```python
# logging_config.py

import logging
import sys
from loguru import logger

# Loguru config (lepsze niż standard logging)
logger.remove()  # Usuń default handler
logger.add(
    sys.stdout,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan> - <level>{message}</level>",
    level="INFO"
)
logger.add(
    "logs/app.log",
    rotation="500 MB",  # Nowy plik co 500MB
    retention="10 days",  # Usuń po 10 dniach
    level="INFO"
)

# Użycie
@app.get("/api/users/{user_id}")
async def get_user(user_id: int):
    logger.info(f"Fetching user {user_id}")
    try:
        user = await db.get_user(user_id)
        logger.debug(f"User {user_id} found: {user.email}")
        return user
    except Exception as e:
        logger.error(f"Error fetching user {user_id}: {str(e)}")
        raise
```

---

### 2. Error Tracking - Sentry

```python
# main.py

import sentry_sdk
from sentry_sdk.integrations.fastapi import FastApiIntegration

sentry_sdk.init(
    dsn="https://your-dsn@sentry.io/project-id",
    integrations=[FastApiIntegration()],
    traces_sample_rate=0.1,  # 10% requestów tracked
    environment="production"
)

# Automatycznie trackuje wszystkie błędy!
```

**Sentry pokazuje:**
- Stack traces
- User context (kto dostał błąd)
- Request data (URL, headers, body)
- Performance metrics

---

### 3. Metrics - Prometheus + Grafana

```python
# metrics.py

from prometheus_fastapi_instrumentator import Instrumentator

# FastAPI metrics
Instrumentator().instrument(app).expose(app)

# Endpoint: GET /metrics (Prometheus scrapes)
```

**Metryki:**
- Request count
- Request duration
- Error rate
- Active requests

**Grafana dashboards** - wizualizacja metryk.

---

## 7. Best Practices - checklist

### ✅ Minimum Viable Production:

- [ ] **Uvicorn z workerami** (`--workers 4`)
- [ ] **Environment variables** (`.env` + `pydantic-settings`)
- [ ] **Docker + docker-compose**
- [ ] **PostgreSQL** (nie SQLite)
- [ ] **Health check endpoint** (`/health`)
- [ ] **Logging** (basic Python logging lub Loguru)
- [ ] **Graceful shutdown** (FastAPI `@app.on_event("shutdown")`)

---

### ✅ Recommended Production:

- [ ] **nginx jako reverse proxy**
- [ ] **HTTPS/SSL** (Let's Encrypt)
- [ ] **Gunicorn + Uvicorn workers** (process management)
- [ ] **Database migrations** (Alembic)
- [ ] **Error tracking** (Sentry)
- [ ] **Monitoring** (Prometheus + Grafana)
- [ ] **Automated backups** (PostgreSQL)
- [ ] **CI/CD pipeline** (GitHub Actions, GitLab CI)
- [ ] **Rate limiting** (nginx lub FastAPI middleware)
- [ ] **CORS configuration** (production domains only)

---

### ✅ Enterprise Production:

- [ ] **Load balancer** (AWS ALB / nginx)
- [ ] **Multiple application instances** (scaling)
- [ ] **Redis** (caching, sessions, Celery tasks)
- [ ] **CDN** dla statycznych plików (CloudFront, Cloudflare)
- [ ] **Database replicas** (read replicas)
- [ ] **Auto-scaling** (Kubernetes / AWS ECS)
- [ ] **Disaster recovery plan**
- [ ] **Security audits** (penetration testing)
- [ ] **WAF** (Web Application Firewall)
- [ ] **DDoS protection** (Cloudflare)

---

## 8. Deployment workflow - krok po kroku

### 1. Przygotowanie serwera (VPS/EC2)

```bash
# Update system
sudo apt update && sudo apt upgrade -y

# Install Docker
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh

# Install Docker Compose
sudo apt install docker-compose-plugin

# Add user to docker group
sudo usermod -aG docker $USER
```

---

### 2. Clone repository

```bash
git clone https://github.com/yourname/yourproject.git
cd yourproject
```

---

### 3. Konfiguracja environment

```bash
# Stwórz .env
cp .env.example .env
nano .env

# .env
POSTGRES_USER=fastapi_user
POSTGRES_PASSWORD=CHANGE_ME_STRONG_PASSWORD
POSTGRES_DB=fastapi_db
SECRET_KEY=CHANGE_ME_RANDOM_STRING_64_CHARS
ENVIRONMENT=production
```

---

### 4. Build i uruchom

```bash
# Build
docker-compose build

# Uruchom w tle
docker-compose up -d

# Sprawdź logi
docker-compose logs -f web

# Sprawdź status
docker-compose ps
```

---

### 5. Database migrations

```bash
# Alembic migrations
docker-compose exec web alembic upgrade head
```

---

### 6. SSL/HTTPS (Let's Encrypt)

```bash
# Uzyskaj certyfikat
docker-compose run --rm certbot certonly \
  --webroot \
  --webroot-path=/var/www/certbot \
  -d example.com \
  --email your@email.com \
  --agree-tos

# Reload nginx
docker-compose restart nginx
```

---

### 7. Test

```bash
# Health check
curl https://example.com/health

# API test
curl https://example.com/api/users

# SSL test
curl -I https://example.com
```

---

### 8. Monitoring

```bash
# Logi
docker-compose logs -f --tail=100 web

# Resource usage
docker stats

# Health checks
docker-compose ps
```

---

## 9. Troubleshooting - najczęstsze problemy

### Problem 1: 502 Bad Gateway (nginx)

**Przyczyny:**
- Uvicorn nie działa
- Zły port w nginx config
- Docker network problem

**Debug:**
```bash
# Sprawdź czy Uvicorn działa
docker-compose logs web

# Sprawdź czy nginx widzi web service
docker-compose exec nginx ping web

# Test connection
curl http://localhost:8000/health  # Z hosta
```

---

### Problem 2: Database connection error

**Przyczyny:**
- PostgreSQL nie uruchomiony
- Złe credentials
- Host nie "db" ale "localhost"

**Fix:**
```bash
# Sprawdź DB
docker-compose exec db psql -U fastapi_user -d fastapi_db -c "SELECT 1"

# DATABASE_URL musi być:
# postgresql://user:pass@db:5432/dbname  # "db" nie "localhost"!
```

---

### Problem 3: SSL certificate errors

**Przyczyny:**
- Certyfikat wygasł (90 dni)
- Certbot nie auto-renew

**Fix:**
```bash
# Manual renewal
docker-compose run --rm certbot renew

# Restart nginx
docker-compose restart nginx
```

---

## 🎯 Podsumowanie całej serii

### Co się nauczyłeś:

**Notebook 1: HTTP i WSGI**
- HTTP Server serwuje statyczne pliki
- WSGI = tłumacz między HTTP a Python (sync)

**Notebook 2: WSGI vs ASGI**
- ASGI = async wersja WSGI
- Async tylko dla I/O (DB, HTTP calls, files)
- 1 worker ASGI = wiele requestów równocześnie

**Notebook 3: Serwery**
- Uvicorn = implementacja ASGI (FastAPI)
- Gunicorn = implementacja WSGI (Django/Flask)
- Standard ≠ Implementacja

**Notebook 4: nginx**
- nginx = HTTP Server + Reverse Proxy
- nginx ≠ Uvicorn (różne warstwy!)
- Po co: SSL, static files, load balancing, security

**Notebook 5: Production Stack**
- Pełny stack: nginx → Uvicorn → FastAPI → PostgreSQL
- SSL z Let's Encrypt
- Monitoring, logging, health checks
- Best practices

---

### Production Stack - finalne zrozumienie:

```
┌─────────────────────────────────────────────┐
│  Klient (Browser)                           │
└───────────────────┬─────────────────────────┘
                    │ HTTPS
                    ↓
┌─────────────────────────────────────────────┐
│  nginx (Warstwa 1)                          │
│  - HTTP Server                              │
│  - Reverse Proxy                            │
│  - SSL Termination                          │
│  - Static Files                             │
└───────────────────┬─────────────────────────┘
                    │ HTTP (localhost)
                    ↓
┌─────────────────────────────────────────────┐
│  Uvicorn (Warstwa 2)                        │
│  - ASGI Server                              │
│  - HTTP → ASGI translation                  │
│  - Python execution                         │
└───────────────────┬─────────────────────────┘
                    │ ASGI calls
                    ↓
┌─────────────────────────────────────────────┐
│  FastAPI (Warstwa 3)                        │
│  - Framework                                │
│  - Routing, Validation, Auth                │
│  - Twój kod Python                          │
└───────────────────┬─────────────────────────┘
                    │ SQL queries
                    ↓
┌─────────────────────────────────────────────┐
│  PostgreSQL (Warstwa 4)                     │
│  - Database                                 │
│  - Persystencja danych                      │
└─────────────────────────────────────────────┘
```

**Każda warstwa ma swoją rolę - razem tworzą production-ready system!**

---

## 📚 Źródła i dalsze nauki:

- [FastAPI Deployment Guide](https://fastapi.tiangolo.com/deployment/)
- [WSGI PEP 3333](https://peps.python.org/pep-3333/)
- [ASGI Specification](https://asgi.readthedocs.io/)
- [Uvicorn Docs](https://www.uvicorn.org/)
- [Gunicorn Docs](https://docs.gunicorn.org/)
- [nginx Docs](https://nginx.org/en/docs/)
- [12-Factor App](https://12factor.net/)
- [Let's Encrypt](https://letsencrypt.org/)
- [Docker Compose](https://docs.docker.com/compose/)

---

**Gratulacje! 🎉**

**Teraz rozumiesz pełny production deployment od HTTP przez WSGI/ASGI, serwery, nginx, aż po kompletny stack z SSL i monitoring!**

---